## PS2 - campus route search (Greedy Best-First vs A*)

weighted bidirectional graph, each node has a heuristic h(n) estimating cost to the goal. Greedy uses f(n)=h(n), A* uses f(n)=g(n)+h(n).

haven't used heapq in a bit, quick refresher before writing the real thing.

In [1]:
import heapq
pq = []
heapq.heappush(pq, (5, 'x'))
heapq.heappush(pq, (2, 'y'))
heapq.heappush(pq, (8, 'z'))
print(heapq.heappop(pq))
print(heapq.heappop(pq))

(2, 'y')
(5, 'x')


right, pops the smallest first tuple element, that's exactly what f(n)=h(n) needs. try greedy on a tiny made up graph before the real campus graph.

In [2]:
import sys
import time
import heapq


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    n, m = map(int, data[idx].split())
    idx += 1

    graph = {}
    for _ in range(m):
        u, v, c = data[idx].split()
        c = int(c)
        graph.setdefault(u, []).append((v, c))
        graph.setdefault(v, []).append((u, c))
        idx += 1

    start, goal = data[idx].split()
    idx += 1

    h = {}
    for _ in range(n):
        node, val = data[idx].split()
        h[node] = int(val)
        idx += 1

    return graph, h, start, goal


def path_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        for nb, c in graph[path[i]]:
            if nb == path[i + 1]:
                total += c
                break
    return total


def greedy_best_first(graph, h, start, goal):
    start_time = time.time()
    closed = set()
    parent = {}
    frontier = [(h[start], start)]
    nodes_expanded = 0

    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, path_cost(graph, path), nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            if nb not in closed:
                if nb not in parent:
                    parent[nb] = cur
                heapq.heappush(frontier, (h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def a_star(graph, h, start, goal):
    start_time = time.time()
    best_g = {start: 0}
    parent = {}
    frontier = [(h[start], start)]
    closed = set()
    nodes_expanded = 0

    while frontier:
        f, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, best_g[goal], nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            new_g = best_g[cur] + cost
            if nb not in best_g or new_g <= best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = cur
                heapq.heappush(frontier, (new_g + h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def print_result(algo_name, found, path, cost, nodes_expanded, exec_time):
    print(f"Algorithm: {algo_name}")
    if not found:
        print("Path Found: No")
        print(f"Nodes Expanded = {nodes_expanded}")
        print(f"Execution Time = {exec_time:.6f}")
        return
    print("Path Found: Yes")
    print("Path: " + " -> ".join(path))
    print(f"Total Cost = {cost}")
    print(f"Nodes Expanded = {nodes_expanded}")
    print(f"Execution Time = {exec_time:.6f}")




In [3]:
toy_graph = {'X':[('Y',1),('Z',10)], 'Y':[('X',1),('Z',1)], 'Z':[('X',10),('Y',1)]}
toy_h = {'X':2, 'Y':1, 'Z':0}
print(greedy_best_first(toy_graph, toy_h, 'X', 'Z'))

(True, ['X', 'Z'], 10, 2, 0.0)


wait that just went straight X -> Z, not through Y like expected. right, Z is directly reachable from X too, and h(goal) is always 0 which beats every other heuristic value automatically, so as soon as Z shows up in the frontier greedy grabs it, doesn't matter that the direct edge costs 10. remove that direct edge so it actually has to pick between two intermediate hops.

In [4]:
toy_graph = {'X':[('Y',1),('W',1)], 'Y':[('X',1),('Z',5)], 'W':[('X',1),('Z',5)], 'Z':[('Y',5),('W',5)]}
toy_h = {'X':2, 'Y':1, 'W':4, 'Z':0}
print(greedy_best_first(toy_graph, toy_h, 'X', 'Z'))

(True, ['X', 'Y', 'Z'], 6, 3, 0.0)


there, goes X -> Y -> Z since Y's heuristic (1) beats W's (4), even though both edges out of X cost the same. that's the behaviour I actually wanted to see. now the real campus graph from the assignment.

In [5]:
graph = {
    'A':[('B',4),('C',2)], 'B':[('A',4),('D',5)],
    'C':[('A',2),('D',3),('E',6)], 'D':[('B',5),('C',3),('E',3)],
    'E':[('C',6),('D',3),('F',2)], 'F':[('E',2)]
}
h = {'A':7,'B':8,'C':5,'D':4,'E':2,'F':0}
print(graph)
print(h)

{'A': [('B', 4), ('C', 2)], 'B': [('A', 4), ('D', 5)], 'C': [('A', 2), ('D', 3), ('E', 6)], 'D': [('B', 5), ('C', 3), ('E', 3)], 'E': [('C', 6), ('D', 3), ('F', 2)], 'F': [('E', 2)]}
{'A': 7, 'B': 8, 'C': 5, 'D': 4, 'E': 2, 'F': 0}


hand check first. from A, neighbors are B(h=8) and C(h=5). greedy always jumps to whichever neighbor has the smallest h, so it goes A -> C. from C, neighbors are D(h=4) and E(h=2), and E is way lower, so greedy should go to E next, not D. see if the code agrees.

In [6]:
print(greedy_best_first(graph, h, 'A', 'F'))

(True, ['A', 'C', 'E', 'F'], 10, 4, 0.0)


so greedy actually goes A -> C -> E -> F, cost 10. matches the hand check, but it's not the same path the assignment PDF shows for greedy (A -> C -> D -> E -> F). cost still comes out to 10 either way since both routes happen to add up the same on this graph (2+6+2 = 2+3+3+2), but the path itself is different. going with what the code actually does since E genuinely has the lower heuristic.

In [7]:
print(a_star(graph, h, 'A', 'F'))

(True, ['A', 'C', 'D', 'E', 'F'], 10, 5, 0.0)


A* gives A -> C -> D -> E -> F, cost 10, matching the PDF's sample output exactly. makes sense, A* tracks the real distance travelled (g) plus the heuristic, so it isn't fooled by a node that just looks close on paper. both A-C-D-E-F and A-C-E-F actually cost 10, so either is a correct optimal path, A* just happened to land on the one the PDF shows.

now build a second graph on purpose where greedy actually loses (the assignment asks for a second test case showing exactly this). idea: put a node right next to the start with a tempting low heuristic but an expensive edge onward, while a slightly less tempting node leads to a cheap route.

In [8]:
graph2 = {'S':[('A',1),('B',2)], 'A':[('S',1),('T',20)], 'B':[('S',2),('T',2)], 'T':[('A',20),('B',2)]}
h2 = {'S':3, 'A':1, 'B':2, 'T':0}
print('greedy:', greedy_best_first(graph2, h2, 'S', 'T'))
print('a star:', a_star(graph2, h2, 'S', 'T'))

greedy: (True, ['S', 'A', 'T'], 21, 3, 0.0)
a star: (True, ['S', 'B', 'T'], 4, 4, 0.0)


there it is. greedy sees A has h=1 (looks closest) and commits, walks S -> A -> T, total cost 21. A* isn't fooled, S -> B -> T is only 4 total even though B's heuristic (2) looked worse than A's (1) at the very first step.

full script that reads stdin and prints in the assignment's format, run for real on the original test case.

In [9]:
import io, sys as _sys
_sys.stdin = io.StringIO('6 7\nA B 4\nA C 2\nB D 5\nC D 3\nC E 6\nD E 3\nE F 2\nA F\nA 7\nB 8\nC 5\nD 4\nE 2\nF 0')
__name__ = '__main__'
import sys
import time
import heapq


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    n, m = map(int, data[idx].split())
    idx += 1

    graph = {}
    for _ in range(m):
        u, v, c = data[idx].split()
        c = int(c)
        graph.setdefault(u, []).append((v, c))
        graph.setdefault(v, []).append((u, c))
        idx += 1

    start, goal = data[idx].split()
    idx += 1

    h = {}
    for _ in range(n):
        node, val = data[idx].split()
        h[node] = int(val)
        idx += 1

    return graph, h, start, goal


def path_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        for nb, c in graph[path[i]]:
            if nb == path[i + 1]:
                total += c
                break
    return total


def greedy_best_first(graph, h, start, goal):
    start_time = time.time()
    closed = set()
    parent = {}
    frontier = [(h[start], start)]
    nodes_expanded = 0

    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, path_cost(graph, path), nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            if nb not in closed:
                if nb not in parent:
                    parent[nb] = cur
                heapq.heappush(frontier, (h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def a_star(graph, h, start, goal):
    start_time = time.time()
    best_g = {start: 0}
    parent = {}
    frontier = [(h[start], start)]
    closed = set()
    nodes_expanded = 0

    while frontier:
        f, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1

        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, best_g[goal], nodes_expanded, time.time() - start_time

        for nb, cost in graph.get(cur, []):
            new_g = best_g[cur] + cost
            if nb not in best_g or new_g <= best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = cur
                heapq.heappush(frontier, (new_g + h[nb], nb))

    return False, [], 0, nodes_expanded, time.time() - start_time


def print_result(algo_name, found, path, cost, nodes_expanded, exec_time):
    print(f"Algorithm: {algo_name}")
    if not found:
        print("Path Found: No")
        print(f"Nodes Expanded = {nodes_expanded}")
        print(f"Execution Time = {exec_time:.6f}")
        return
    print("Path Found: Yes")
    print("Path: " + " -> ".join(path))
    print(f"Total Cost = {cost}")
    print(f"Nodes Expanded = {nodes_expanded}")
    print(f"Execution Time = {exec_time:.6f}")


def main():
    graph, h, start, goal = read_input()

    g_found, g_path, g_cost, g_nodes, g_time = greedy_best_first(graph, h, start, goal)
    print_result("Greedy Best-First Search", g_found, g_path, g_cost, g_nodes, g_time)

    print()

    a_found, a_path, a_cost, a_nodes, a_time = a_star(graph, h, start, goal)
    print_result("A* Search", a_found, a_path, a_cost, a_nodes, a_time)

    print()
    print("Comparison:")
    if g_found and a_found:
        print(f"Path cost -> Greedy = {g_cost}, A* = {a_cost}")
    print(f"Nodes expanded -> Greedy = {g_nodes}, A* = {a_nodes}")
    print(f"Execution time -> Greedy = {g_time:.6f}, A* = {a_time:.6f}")
    print("A* is optimal since it accounts for both the cost so far (g) and the")
    print("estimated cost to goal (h). Greedy only looks at h, so it can walk into")
    print("a locally attractive but globally worse route.")


if __name__ == "__main__":
    main()


Algorithm: Greedy Best-First Search
Path Found: Yes
Path: A -> C -> E -> F
Total Cost = 10
Nodes Expanded = 4
Execution Time = 0.000000

Algorithm: A* Search
Path Found: Yes
Path: A -> C -> D -> E -> F
Total Cost = 10
Nodes Expanded = 5
Execution Time = 0.000000

Comparison:
Path cost -> Greedy = 10, A* = 10
Nodes expanded -> Greedy = 4, A* = 5
Execution time -> Greedy = 0.000000, A* = 0.000000
A* is optimal since it accounts for both the cost so far (g) and the
estimated cost to goal (h). Greedy only looks at h, so it can walk into
a locally attractive but globally worse route.
